<a href="https://colab.research.google.com/github/Lynna5639582/Revision-Templates/blob/main/Classic_ML_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Logistic Regression, Decision Tree, Random Forest, GBDT

Load & Inspect Data

In [ ]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('loan_data.csv') *change the file name
print(df.head())
print('Shape:', df.shape)
df.info()
print('Missing values:', df.isna().sum().sum())
print('Target counts:')
print(df['loan_status'].value_counts())

Separate Predictors and Target





In [ ]:
# To prepare our features (X), we drop the target column 'num'
X = df.drop(columns=['num'])
y = df['num']
print('X:', X.shape, 'y:', y.shape)

Split & Scale

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=4567, stratify=y
)
#stratify=y makes sure that the proportion of each class in our target variable is approximately the same in both the training and test sets.
print('Training:', X_train.shape, y_train.shape)
print('Test:', X_test.shape, y_test.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Ensure we use the encoded features
X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

# Split using the encoded dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=4567, stratify=y
)

# Scale the features using training data ONLY
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Scaled training shape:', X_train_scaled.shape)
print('Scaled test shape:', X_test_scaled.shape)

Train & Compare Models (Train vs. Test Table)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

#max_iter specifies the maximum number of iterations the optimization algorithm can perform when training the model
models = {
    #random_state=4567 — Random seed
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=4567),
    #n_estimators=100: Number of trees
    #random_state=4567 makes the results reproducible.
    #n_jobs controls how many CPU cores can be used for parallel computation
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=4567, n_jobs=-1),
    #n_estimators=80 means 80 boosting stages.
    'GBDT': GradientBoostingClassifier(n_estimators=80, random_state=4567),
}
for name, model in models.items():
    training_features = X_train_scaled if name == 'Logistic Regression' else X_train
    model.fit(training_features, y_train)
    print(name, 'trained')

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

results = []
for name, model in models.items():
    for split, features, target in [
        ('Train', X_train_scaled if name == 'Logistic Regression' else X_train, y_train),
        ('Test', X_test_scaled if name == 'Logistic Regression' else X_test, y_test),
    ]:
        pred = model.predict(features)
        precision, recall, f1, _ = precision_recall_fscore_support(
            target, pred, average='macro', zero_division=0
        )
        results.append({'Model': name, 'Split': split,
                        'Accuracy': accuracy_score(target, pred),
                        'Macro Precision': precision, 'Macro Recall': recall,
                        'Macro F1': f1})
results_df = pd.DataFrame(results)
display(results_df.round(3))

Confusion Matrix & Feature Importance

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

best_model = models['Random Forest']  # <-- EDIT: pick winning model
test_features = X_test_scaled if best_model == models['Logistic Regression'] else X_test

ConfusionMatrixDisplay.from_predictions(y_test, best_model.predict(test_features))
plt.title("Confusion Matrix - Best Model")
plt.show()